In [ ]:
import pandas as pd
from datetime import datetime

# Đọc file CSV
df = pd.read_csv("C:/Users/hello/Downloads/capstone (1)/capstone/data/hr_dashboard_data.csv")

# 2. CLEANING

#  2.1 Clean Joining Date (handle missing year do CSV) 
def parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    date_str = str(date_str)

    parts = date_str.split("-")

    # Case 1: "20-Jan" → thiếu năm
    if len(parts) == 2 and parts[0].isdigit():
        date_str = f"{date_str}-{datetime.now().year}"
        return pd.to_datetime(date_str, format="%d-%b-%Y", errors='coerce')

    # Case 2: "Jan-99"
    elif len(parts) == 2:
        return pd.to_datetime(date_str, format="%b-%y", errors='coerce')

    # Case 3: fallback (nếu có format khác)
    return pd.to_datetime(date_str, errors='coerce')


df["Joining Date"] = df["Joining Date"].apply(parse_date)

#  2.2 Handle missing Salary 
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

#  2.3 Remove duplicates based on Name 
df = df.drop_duplicates(subset="Name")


# 3. FEATURE ENGINEERING


#  3.1 Experience Years 
current_year = datetime.now().year
df["Experience Years"] = current_year - df["Joining Date"].dt.year

#  3.2 Productivity Group 
def productivity_group(x):
    if x < 50:
        return "Low"
    elif 50 <= x <= 80:
        return "Medium"
    else:
        return "High"

df["Productivity Group"] = df["Productivity (%)"].apply(productivity_group)


# 4. CHECK DATA

print(df.info())
print(df.head())
print(df.isnull().sum())


# 5. SAVE CLEAN DATA
df.to_csv("cleaned_hr_data.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Name                   200 non-null    object        
 1   Age                    200 non-null    int64         
 2   Gender                 200 non-null    object        
 3   Projects Completed     200 non-null    int64         
 4   Productivity (%)       200 non-null    int64         
 5   Satisfaction Rate (%)  200 non-null    int64         
 6   Feedback Score         200 non-null    float64       
 7   Department             200 non-null    object        
 8   Position               200 non-null    object        
 9   Joining Date           200 non-null    datetime64[ns]
 10  Salary                 200 non-null    int64         
 11  Experience Years       200 non-null    int32         
 12  Productivity Group     200 non-null    object        
dtypes: da